# EXP-C — repeated-image caching

One fixed 896 px image (1026 vision tokens — the expensive case), 20 distinct questions, three configurations:

| config | prefix caching | prompt order | expectation |
|---|---|---|---|
| `apc_off` | disabled | image first | every request pays full price |
| `apc_on_image_first` | enabled | image first | requests 2–20 reuse the image's KV blocks |
| `apc_on_question_first` | enabled | question first | **no** benefit — the changing question breaks the shared prefix before the image |

Config 1 and configs 2–3 run on separate server processes, so no cache state leaks.
Runtime → **T4 GPU**. Total ≈ 30 min.


In [ ]:
!nvidia-smi -L


In [ ]:
# Install vLLM 0.25.1 built for CUDA 12.9 (PyPI default targets CUDA 13 and fails on Colab)
%pip -q install https://github.com/vllm-project/vllm/releases/download/v0.25.1/vllm-0.25.1+cu129-cp38-abi3-manylinux_2_28_x86_64.whl --extra-index-url https://download.pytorch.org/whl/cu129
import vllm, transformers
print('vllm', vllm.__version__, '| transformers', transformers.__version__)


## Benchmark code (mirrors `bench/` in the repo)


In [ ]:
import base64, io, json, subprocess, time
import numpy as np
import requests
from PIL import Image
from transformers import AutoTokenizer

RESULTS = '/content/results_exp_c.jsonl'

import base64
import io

import numpy as np
from PIL import Image


def image_tokens(width: int, height: int) -> int:
    """Tokens an image costs in Qwen2-VL: one per 28x28 pixel block."""
    return (height // 28) * (width // 28)


def noise_image(side_px: int, seed: int) -> Image.Image:
    """A square RGB random-noise image. Noise (not a flat color) so the JPEG
    is realistically sized; a distinct seed gives a distinct image, which
    defeats prefix caching in cold experiments."""
    if side_px % 28 != 0:
        raise ValueError(f"side_px must be a multiple of 28, got {side_px}")
    rng = np.random.default_rng(seed)
    pixels = rng.integers(0, 256, size=(side_px, side_px, 3), dtype=np.uint8)
    return Image.fromarray(pixels)


def to_data_url(img: Image.Image) -> str:
    """Encode an image as the data-URL string the OpenAI-style API expects."""
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/jpeg;base64,{b64}"


def text_prompt(target_tokens: int, tokenizer, seed: int) -> str:
    """A unique string that tokenizes to exactly target_tokens tokens
    (measured without special tokens; the chat template adds its own).

    Build over-long text, trim the token ids, decode back — then repair,
    because decode/encode is not a perfect round trip and the count can
    shift by a token."""
    text = f"case {seed}:" + " benchmark" * target_tokens
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) < target_tokens:
        raise ValueError(f"filler too short for {target_tokens} tokens")
    text = tokenizer.decode(ids[:target_tokens])

    for _ in range(50):
        count = len(tokenizer.encode(text, add_special_tokens=False))
        if count == target_tokens:
            return text
        if count > target_tokens:
            ids = tokenizer.encode(text, add_special_tokens=False)[:-1]
            text = tokenizer.decode(ids)
        else:
            text += " ok"
    raise RuntimeError(f"could not converge on {target_tokens} tokens")


def text_payload(model: str, prompt: str, max_tokens: int = 64) -> dict:
    """Request body for a text-only chat completion. temperature=0 so the
    same input always gives the same output (reproducible benchmarks)."""
    return {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0,
        "max_tokens": max_tokens,
    }


def image_payload(model: str, data_url: str, question: str, max_tokens: int = 64,
                  image_first: bool = False) -> dict:
    """Request body for an image + question chat completion.

    image_first matters for prefix caching: the cache reuses a shared
    *prefix*, so repeated content (the image) must come before the part
    that changes (the question) or no request can ever hit the cache."""
    text_block = {"type": "text", "text": question}
    image_block = {"type": "image_url", "image_url": {"url": data_url}}
    content = [image_block, text_block] if image_first else [text_block, image_block]
    return {
        "model": model,
        "messages": [{"role": "user", "content": content}],
        "temperature": 0,
        "max_tokens": max_tokens,
    }

import json
import time

import requests


def send_request(payload: dict, base_url: str = "http://localhost:8000") -> dict:
    """POST one chat completion and stream the reply. Returns timings,
    the server-reported prompt token count, and the full output text."""
    payload = dict(payload, stream=True, stream_options={"include_usage": True})

    t_start = time.perf_counter()
    ttft = None
    parts = []
    usage = None

    url = f"{base_url}/v1/chat/completions"
    with requests.post(url, json=payload, stream=True, timeout=600) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if not line.startswith(b"data: "):
                continue
            data = line[len(b"data: "):]
            if data == b"[DONE]":
                break
            chunk = json.loads(data)
            if chunk.get("usage"):
                usage = chunk["usage"]
            for choice in chunk.get("choices", []):
                content = choice.get("delta", {}).get("content")
                if content:
                    if ttft is None:
                        ttft = time.perf_counter() - t_start
                    parts.append(content)

    return {
        "ttft_s": ttft,
        "e2e_s": time.perf_counter() - t_start,
        "prompt_tokens": usage["prompt_tokens"] if usage else None,
        "completion_tokens": usage["completion_tokens"] if usage else None,
        "output_text": "".join(parts),
    }

import subprocess
import time

import requests

MODEL = "Qwen/Qwen2-VL-2B-Instruct"


def build_args(model: str = MODEL, apc: bool = True, port: int = 8000) -> list:
    args = [
        "vllm", "serve", model,
        "--dtype", "half",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85",
        "--enforce-eager",
        "--limit-mm-per-prompt", '{"image": 1}',
        "--mm-processor-kwargs", '{"min_pixels": 784, "max_pixels": 1003520}',
        "--port", str(port),
    ]
    if not apc:
        args.append("--no-enable-prefix-caching")
    return args


def start(model: str = MODEL, apc: bool = True, port: int = 8000,
          log_path: str = "/content/vllm.log"):
    log_file = open(log_path, "w")
    return subprocess.Popen(build_args(model, apc, port),
                            stdout=log_file, stderr=subprocess.STDOUT)


def wait_ready(proc, base_url: str = "http://localhost:8000",
               log_path: str = "/content/vllm.log", timeout_s: int = 600) -> None:
    for _ in range(timeout_s // 5):
        if proc.poll() is not None:
            raise RuntimeError("server exited:\n" + open(log_path).read()[-3000:])
        try:
            if requests.get(f"{base_url}/v1/models", timeout=2).ok:
                return
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(5)
    raise RuntimeError(f"server not ready within {timeout_s}s")


def stop(proc) -> None:
    proc.terminate()
    proc.wait(timeout=30)

import json



def append_jsonl(path: str, record: dict) -> None:
    with open(path, "a") as f:
        f.write(json.dumps(record) + "\n")


def warmup(model: str, tokenizer, base_url: str = "http://localhost:8000") -> None:
    """Throwaway requests so one-time costs (Triton kernel compilation,
    per-shape vision kernels) are paid before any measured request."""
    print("warmup: text")
    send_request(text_payload(model, text_prompt(64, tokenizer, seed=999_000), max_tokens=8), base_url)
    for side in (224, 448, 672, 896):
        print(f"warmup: image {side}px")
        payload = image_payload(model, to_data_url(noise_image(side, seed=999_000 + side)), "Describe.", max_tokens=8)
        send_request(payload, base_url)


def run_text_sweep(model, tokenizer, targets, runs_per_target, out_path,
                   base_url: str = "http://localhost:8000") -> None:
    """EXP-T: unique text prompts at each target token count."""
    seed = 0
    for target in targets:
        for run in range(runs_per_target):
            seed += 1
            prompt = text_prompt(target, tokenizer, seed)
            result = send_request(text_payload(model, prompt, max_tokens=16), base_url)
            append_jsonl(out_path, {
                "experiment": "EXP-T",
                "target_tokens": target,
                "seed": seed,
                "prompt_tokens": result["prompt_tokens"],
                "ttft_s": result["ttft_s"],
                "e2e_s": result["e2e_s"],
            })
            print(f"EXP-T {target:>4} tok  run {run + 1}/{runs_per_target}  "
                  f"prompt_tokens={result['prompt_tokens']}  ttft={result['ttft_s']:.3f}s")


def run_image_sweep(model, sides, runs_per_side, out_path,
                    base_url: str = "http://localhost:8000") -> None:
    """EXP-V: unique noise images at each resolution."""
    seed = 0
    for side in sides:
        for run in range(runs_per_side):
            seed += 1
            data_url = to_data_url(noise_image(side, seed))
            payload = image_payload(model, data_url, "Describe this image.", max_tokens=16)
            result = send_request(payload, base_url)
            append_jsonl(out_path, {
                "experiment": "EXP-V",
                "side_px": side,
                "seed": seed,
                "prompt_tokens": result["prompt_tokens"],
                "ttft_s": result["ttft_s"],
                "e2e_s": result["e2e_s"],
            })
            print(f"EXP-V {side:>3}px  run {run + 1}/{runs_per_side}  "
                  f"prompt_tokens={result['prompt_tokens']}  ttft={result['ttft_s']:.3f}s")


def run_repeated_image(model, side, image_seed, n_questions, out_path, config,
                       image_first, base_url: str = "http://localhost:8000") -> None:
    """EXP-C: one fixed image, many distinct questions. Request 1 is always
    cold; whether 2..n go faster depends on the cache config and on whether
    the image precedes the changing question in the prompt."""
    data_url = to_data_url(noise_image(side, image_seed))
    for i in range(1, n_questions + 1):
        question = f"Question {i}: describe aspect number {i} of this image."
        payload = image_payload(model, data_url, question, max_tokens=16,
                                image_first=image_first)
        result = send_request(payload, base_url)
        append_jsonl(out_path, {
            "experiment": "EXP-C",
            "config": config,
            "request_index": i,
            "side_px": side,
            "prompt_tokens": result["prompt_tokens"],
            "ttft_s": result["ttft_s"],
            "e2e_s": result["e2e_s"],
        })
        print(f"EXP-C [{config}] req {i:>2}/{n_questions}  "
              f"prompt_tokens={result['prompt_tokens']}  ttft={result['ttft_s']:.3f}s")

tokenizer = AutoTokenizer.from_pretrained(MODEL)
print('benchmark code loaded')


## Config 1 — caching disabled (baseline)


In [ ]:
proc = start(apc=False)
print('server starting (apc OFF), pid', proc.pid)
wait_ready(proc)
print('ready')
warmup(MODEL, tokenizer)
run_repeated_image(MODEL, side=896, image_seed=42, n_questions=20,
                   out_path=RESULTS, config='apc_off', image_first=True)
stop(proc)
print('config 1 done, server stopped')


## Configs 2 + 3 — caching enabled

Fresh server. Config 2 sends the image **before** the question; config 3 sends a *different* fixed image **after** the question. Same cache, opposite outcomes — if the ordering theory is right.


In [ ]:
proc = start(apc=True)
print('server starting (apc ON), pid', proc.pid)
wait_ready(proc)
print('ready')
warmup(MODEL, tokenizer)
run_repeated_image(MODEL, side=896, image_seed=42, n_questions=20,
                   out_path=RESULTS, config='apc_on_image_first', image_first=True)
run_repeated_image(MODEL, side=896, image_seed=43, n_questions=20,
                   out_path=RESULTS, config='apc_on_question_first', image_first=False)


In [ ]:
# What the server counted: prefix-cache stats from the log
!grep -iE 'prefix cache|hit rate' /content/vllm.log | tail -10


## Analysis


In [ ]:
import numpy as np

import json

import numpy as np


def load_jsonl(path: str) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def fit_line(records: list[dict]) -> tuple[float, float]:
    """Least-squares TTFT vs prompt_tokens.
    Returns (slope in ms per token, intercept in ms)."""
    x = np.array([r["prompt_tokens"] for r in records], dtype=float)
    y = np.array([r["ttft_s"] for r in records], dtype=float) * 1000.0
    slope, intercept = np.polyfit(x, y, 1)
    return float(slope), float(intercept)


def median_ttft_by(records: list[dict], key: str) -> dict:
    """Median TTFT (seconds) per group, e.g. per target_tokens or side_px."""
    groups: dict = {}
    for r in records:
        groups.setdefault(r[key], []).append(r["ttft_s"])
    return {k: float(np.median(v)) for k, v in sorted(groups.items())}


def first_occurrence(records: list[dict], keys: tuple[str, ...]) -> list[dict]:
    """Keep only the first record per key combination. Guards analysis
    against a sweep cell accidentally run twice: repeated inputs are no
    longer cold, so only the first pass counts as a cold measurement."""
    seen = set()
    kept = []
    for r in records:
        key = tuple(r.get(k) for k in keys)
        if key not in seen:
            seen.add(key)
            kept.append(r)
    return kept

records = [r for r in load_jsonl(RESULTS) if r['experiment'] == 'EXP-C']
print(f'{"config":<24} {"cold (req 1)":>12} {"warm (med 2-20)":>16} {"speedup":>8}')
summary = {}
for config in ('apc_off', 'apc_on_image_first', 'apc_on_question_first'):
    rows = [r for r in records if r['config'] == config]
    cold = rows[0]['ttft_s']
    warm = float(np.median([r['ttft_s'] for r in rows[1:]]))
    summary[config] = (cold, warm)
    print(f'{config:<24} {cold*1000:>10.0f}ms {warm*1000:>14.0f}ms {cold/warm:>7.1f}x')


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
labels = ['caching off\n(image first)', 'caching on\nimage first', 'caching on\nquestion first']
configs = ['apc_off', 'apc_on_image_first', 'apc_on_question_first']
x = np.arange(len(configs))
cold = [summary[c][0] * 1000 for c in configs]
warm = [summary[c][1] * 1000 for c in configs]
ax.bar(x - 0.18, cold, 0.32, label='cold (request 1)', color='#8B95A0')
ax.bar(x + 0.18, warm, 0.32, label='warm (median of 2-20)', color='#C2410C')
ax.set_xticks(x, labels)
ax.set_ylabel('TTFT (ms)')
ax.set_title('Repeated-image TTFT - one 896px image, 20 distinct questions')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.25)
fig.savefig('/content/exp_c_caching.png', dpi=150, bbox_inches='tight')
plt.show()


## Done

1. Paste the stats-grep output and the analysis table back to the Claude session.
2. Download `results_exp_c.jsonl` and `exp_c_caching.png` into the repo's `results/` folder.
